# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

I pick **ranking** (also called scoring). The question is "which ones first?" — I need to rank content pages by how likely they are to be declining, so an editor can review the top ones first. That is a classic ranking task: every page gets a score, and the score determines the order.

Classification (yes/no per page) could work too, but the real decision is about **priority** — the editor does not fix all declining pages at once, they pick the top batch. A ranking score lets me say "these 50 are the most worth looking at right now."

In [ ]:
# Show why ranking matters: the editor does not have time for all pages.
import os
import pandas as pd

def locate_csv():
    roots = ['/content', './', '../', '../../data/raw', '/']
    for r in roots:
        for dirpath, _, files in os.walk(r):
            if 'content_refresh_anonymized.csv' in files:
                return os.path.join(dirpath, 'content_refresh_anonymized.csv')
    return None

path = locate_csv()
if path is None:
    raise FileNotFoundError('CSV not found. Upload content_refresh_anonymized.csv to Colab.')
print('Using:', path)

df = pd.read_csv(path)
total = len(df)
declining = (df['trend_direction'] == 'down').sum()
print(f'{total:,} pages total, {declining:,} declining ({declining/total:.1%}).')
print(f'An editor can review maybe 50 pages per sprint. Ranking which 50 matters.')

## 2. Target or proxy

The target is `is_declining_label` — 1 when `trend_direction == "down"`, else 0. This is an **observed** outcome: it comes from comparing actual impressions in the last 30 days versus the 30 days before that. It is not a rule someone made up — it is measured from real search data.

A page is labelled declining when its impressions dropped by more than 20% between the two windows. That is a real signal the data already measured, not a prediction.

In [ ]:
# Check the label distribution.
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)
label_rate = df['is_declining_label'].mean()
print(f'Label distribution: {df["is_declining_label"].value_counts().to_dict()}')
print(f'Base rate (declining share): {label_rate:.3f}')
print(f'A random guess is right {label_rate:.1%} of the time — that is the number to beat.')

## 3. Success metric

I pick **Precision@50**: of the top 50 pages a ranking flags, how many are actually declining.

Why Precision@50? The editor reviews the list in order. If the top 50 contains 30 real declining pages, that is a useful list. If it contains only 15, the editor wastes half the sprint on pages that did not need attention. Precision@50 directly measures how useful the ranked list is to the person who acts on it.

The random baseline is the base rate (~0.54). A good ranking should put far more declining pages in the top 50 than random chance.

In [ ]:
# Compute the random-baseline precision@50 for comparison.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y = df['is_declining_label'].values

# Random baseline: if you pick 50 pages at random, what precision do you get?
np.random.seed(42)
random_scores = np.random.rand(len(df))
random_p50 = precision_at_k(random_scores, y, 50)

print(f'Random baseline Precision@50: {random_p50:.3f} (should be close to base rate {label_rate:.3f})')
print(f'A useful ranking needs to beat this number clearly.')

## 4. The unit of analysis, as a real dataframe

One row = one content page. Each page belongs to one client, has measured search signals (impressions, clicks, position, ctr), engagement signals (sessions, scroll, ai traffic), and content properties (age, word count, freshness). The label tells us whether that page's impressions are declining.

This is the right grain because the decision is per-page: "should we refresh THIS page?"

In [ ]:
# Show the dataframe shape and a small sample.
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'Unique clients: {df["client_id"].nunique()}')
print()

# Key columns the model will use — pre-decision signals only, no product flags.
key_cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d',
            'avg_position', 'ctr', 'content_age_days', 'days_since_last_update',
            'word_count', 'engagement_rate', 'scroll_rate', 'is_declining_label']
df[key_cols].head(5)

## 5. Why ML beats a fixed rule here

A hand rule like "stale AND visible" (not updated in 180+ days and getting 500+ impressions) is a decent first cut — it catches some declining pages. But it only looks at two signals. The real pattern is messier: a page might be young but dropping fast in ctr, or have high impressions but a terrible avg_position that is slipping. There are 10+ signals tangled together, and their importance shifts depending on the content type and the client.

An if-statement cannot weigh all of those at once. A model can — and it can show you exactly which signal it leaned on (that is Week 2's guided notebook). The ranking it produces should put more declining pages in the editor's top 50 than any hand-written threshold.

In [ ]:
# Quick look at why a single rule misses: how many declining pages are NOT stale+visible?
stale = df['days_since_last_update'] >= 180
visible = df['impressions_90d'] >= 500
hand_rule_caught = (stale & visible & (df['is_declining_label'] == 1)).sum()
total_declining = df['is_declining_label'].sum()

print(f'Declining pages caught by stale+visible rule: {hand_rule_caught:,} out of {total_declining:,}')
print(f'That means {total_declining - hand_rule_caught:,} declining pages are missed by the simple rule.')
print(f'A model using all signals should catch more of them in the top 50.')

## Self-check

Before I submit, I confirm each line honestly:

- [ ] Every section above is filled with my words AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then I submit my repo URL on the card. Done.